# SurfaceTarget-CRC — phase 05 — patch 5 2

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.2
#
# BIOLOGICAL TARGET CLASSIFICATION
# AND EVIDENCE ELIGIBILITY AUDIT
#
# OBJECTIVES:
# 1. Validate the canonical Phase 5 evidence table
# 2. Classify protein topology evidence
# 3. Distinguish GPI-anchored and transmembrane proteins
# 4. Preserve uncertainty about antibody accessibility
# 5. Classify CPTAC protein-evidence completeness
# 6. Examine HPA normal colon, kidney and skin staining
# 7. Preserve normal-tissue IHC reliability information
# 8. Identify candidates requiring biological review
# 9. Review the 20 cross-omics-supported candidates
# 10. Examine leading membrane-related candidates
# 11. Save classification tables and provenance
# 12. Create a research checkpoint
#
# IMPORTANT:
# - No therapeutic efficacy claims
# - No clinical safety scores
# - No candidate exclusions
# - No arbitrary target-ranking weights
# - Missing evidence is not negative evidence
# - Bulk expression does not identify malignant cell origin
# - Membrane localization does not prove antibody binding
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [META, RESULTS, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


CANONICAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_integrated_evidence_v1.csv"
)

TISSUE_PATH = (
    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv"
)

PREVIOUS_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_integration_provenance.json"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.2")
print("BIOLOGICAL TARGET CLASSIFICATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VERIFY SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    CANONICAL_PATH,
    TISSUE_PATH,
    PREVIOUS_PROVENANCE_PATH
]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PREVIOUS_PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


expected_hash = previous_provenance.get(
    "canonical_output_sha256"
)

actual_hash = sha256_file(
    CANONICAL_PATH
)


if not expected_hash:

    raise RuntimeError(
        "Canonical checksum missing from Patch 5.1 provenance."
    )


if actual_hash != expected_hash:

    raise RuntimeError(
        "Phase 5 canonical evidence checksum mismatch."
    )


print("\nCanonical evidence checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD CANONICAL EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANONICAL CROSS-OMICS EVIDENCE")
print("-" * 65)


source = pd.read_csv(
    CANONICAL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "protein_enriched",
    "mss_complete_pairs",
    "uniprot_topology_category",
    "uniprot_gpi_anchor",
    "uniprot_reviewed_available",
    "phase5_normal_ihc_available",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues",
    "depmap_final_mapping_available",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}


missing_columns = (
    required_columns - set(source.columns)
)


if missing_columns:

    raise RuntimeError(
        f"Required evidence columns missing: "
        f"{sorted(missing_columns)}"
    )


if len(source) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if source["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene identifiers."
    )


if source["ensembl_stable_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )


print("\nCanonical candidates:", len(source))

print("Evidence columns:", len(source.columns))


# ------------------------------------------------------------
# 4. PREPARE STRICT BOOLEAN INTERPRETATION
# ------------------------------------------------------------

def parse_boolean(series, column_name):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true",
        "false",
        "1",
        "0"
    }

    missing_tokens = {
        "",
        "nan",
        "none",
        "<na>"
    }

    invalid = values[
        values.notna()
        & ~values.isin(
            list(accepted | missing_tokens)
        )
    ]

    if not invalid.empty:

        raise RuntimeError(
            f"Unexpected boolean values in "
            f"{column_name}: "
            f"{invalid.unique().tolist()[:10]}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


data = source.copy(deep=True)


data["_protein_enriched"] = parse_boolean(
    data["protein_enriched"],
    "protein_enriched"
)


data["_uniprot_gpi"] = parse_boolean(
    data["uniprot_gpi_anchor"],
    "uniprot_gpi_anchor"
)


data["_uniprot_reviewed"] = parse_boolean(
    data["uniprot_reviewed_available"],
    "uniprot_reviewed_available"
)


data["_normal_ihc_available"] = parse_boolean(
    data["phase5_normal_ihc_available"],
    "phase5_normal_ihc_available"
)


data["_depmap_available"] = parse_boolean(
    data["depmap_final_mapping_available"],
    "depmap_final_mapping_available"
)


if int(data["_protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 CPTAC-supported candidates."
    )


if int(data["_uniprot_reviewed"].sum()) != 944:

    raise RuntimeError(
        "Expected 944 reviewed UniProt mappings."
    )


if int(data["_normal_ihc_available"].sum()) != 690:

    raise RuntimeError(
        "Expected 690 HPA normal-IHC annotated genes."
    )


if int(data["_depmap_available"].sum()) != 943:

    raise RuntimeError(
        "Expected 943 verified DepMap mappings."
    )


print("\nBoolean evidence validation: PASSED")


# ------------------------------------------------------------
# 5. CLASSIFY UNIPROT TOPOLOGY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING PROTEIN TOPOLOGY")
print("-" * 65)


def classify_topology(row):

    category = str(
        row["uniprot_topology_category"]
    ).strip().lower()

    gpi = bool(
        row["_uniprot_gpi"]
    )

    if (
        "cell membrane with annotated "
        "extracellular region" in category
    ):

        return (
            "Membrane with annotated "
            "extracellular region"
        )

    if gpi and (
        "cell membrane" in category
        or "cell-membrane" in category
    ):

        return (
            "GPI-anchored membrane-associated"
        )

    if gpi:

        return (
            "GPI anchor annotated; "
            "membrane residency requires review"
        )

    if (
        "cell membrane" in category
        or "cell-membrane" in category
    ):

        return (
            "Membrane-associated; "
            "extracellular topology unresolved"
        )

    if (
        "transmembrane protein" in category
        and "surface localization unresolved"
        in category
    ):

        return (
            "Transmembrane; "
            "surface localization unresolved"
        )

    if (
        "secreted or extracellular" in category
    ):

        return (
            "Secreted/extracellular; "
            "membrane attachment not established"
        )

    if (
        "no qualifying cell-surface topology"
        in category
    ):

        return (
            "No qualifying surface-topology "
            "evidence in extracted annotations"
        )

    return (
        "Other or insufficient topology annotation"
    )


data["phase5_topology_class"] = (
    data.apply(
        classify_topology,
        axis=1
    )
)


print("\nTopology class distribution:")

print(
    data["phase5_topology_class"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 6. ASSIGN DESCRIPTIVE SURFACE-REVIEW GROUPS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING SURFACE-EVIDENCE REVIEW GROUPS")
print("-" * 65)


def surface_review_group(category):

    if category == (
        "Membrane with annotated "
        "extracellular region"
    ):

        return (
            "Membrane evidence: "
            "extracellular region annotated"
        )

    if category == (
        "GPI-anchored membrane-associated"
    ):

        return (
            "Membrane evidence: GPI anchor"
        )

    if category in [
        "Membrane-associated; "
        "extracellular topology unresolved",

        "GPI anchor annotated; "
        "membrane residency requires review"
    ]:

        return (
            "Membrane evidence: "
            "accessibility requires review"
        )

    if category == (
        "Transmembrane; "
        "surface localization unresolved"
    ):

        return (
            "Transmembrane evidence: "
            "surface residency unresolved"
        )

    if category == (
        "Secreted/extracellular; "
        "membrane attachment not established"
    ):

        return (
            "Extracellular non-membrane review"
        )

    return (
        "No established surface-targeting "
        "topology from current annotations"
    )


data["phase5_surface_review_group"] = (
    data["phase5_topology_class"]
    .map(surface_review_group)
)


print("\nSurface-review group distribution:")

print(
    data["phase5_surface_review_group"]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "These are evidence groups, "
    "not therapeutic target rankings."
)

print(
    "Membrane annotation does not establish "
    "antibody binding, epitope accessibility "
    "or internalization."
)


# ------------------------------------------------------------
# 7. PRESERVE HPA PLASMA-MEMBRANE EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING INDEPENDENT HPA LOCALIZATION")
print("-" * 65)


if "hpa_subcellular_record_available" in data.columns:

    hpa_localization_available = parse_boolean(
        data["hpa_subcellular_record_available"],
        "hpa_subcellular_record_available"
    )

else:

    hpa_localization_available = pd.Series(
        False,
        index=data.index
    )


if "hpa_plasma_membrane_reported" in data.columns:

    hpa_plasma_membrane = parse_boolean(
        data["hpa_plasma_membrane_reported"],
        "hpa_plasma_membrane_reported"
    )

else:

    hpa_plasma_membrane = pd.Series(
        False,
        index=data.index
    )


data["phase5_hpa_plasma_membrane_status"] = np.select(

    [
        hpa_plasma_membrane.to_numpy(dtype=bool),

        (
            hpa_localization_available
            & ~hpa_plasma_membrane
        ).to_numpy(dtype=bool)
    ],

    [
        "Plasma membrane reported",

        "Localization record available; "
        "plasma membrane not reported"
    ],

    default="HPA localization record unavailable"
)


print("\nHPA localization status:")

print(
    data["phase5_hpa_plasma_membrane_status"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 8. CLASSIFY CPTAC EVIDENCE COMPLETENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING CPTAC PROTEIN EVIDENCE")
print("-" * 65)


data["_mss_complete_pairs"] = pd.to_numeric(
    data["mss_complete_pairs"],
    errors="coerce"
)


def protein_evidence_state(row):

    if row["_protein_enriched"]:

        return (
            "Qualifying MSS paired-protein enrichment"
        )

    n = row["_mss_complete_pairs"]

    if pd.isna(n):

        return (
            "Paired-protein coverage unavailable"
        )

    if n >= 30:

        return (
            "Adequately covered in paired assay; "
            "enrichment criteria not met"
        )

    if n > 0:

        return (
            "Sparse paired-protein measurements"
        )

    return (
        "No complete MSS paired-protein measurement"
    )


data["phase5_protein_evidence_state"] = (
    data.apply(
        protein_evidence_state,
        axis=1
    )
)


print("\nCPTAC evidence categories:")

print(
    data["phase5_protein_evidence_state"]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "Absence of qualifying protein enrichment "
    "does not demonstrate protein absence."
)


# ------------------------------------------------------------
# 9. PREPARE NORMAL-TISSUE EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


normal_columns = [
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


for column in normal_columns:

    data["_" + column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )


def normal_ihc_state(row):

    if not row["_normal_ihc_available"]:

        return (
            "Normal-IHC evidence unavailable"
        )

    high = row["_hpa_tissues_high"]

    medium_high = row[
        "_hpa_tissues_medium_high"
    ]

    if pd.notna(high) and high > 0:

        return (
            "High staining reported "
            "in at least one normal tissue"
        )

    if (
        pd.notna(medium_high)
        and medium_high > 0
    ):

        return (
            "Medium staining reported; "
            "no high tissue-level staining observed"
        )

    return (
        "No medium/high staining in "
        "available summarized IHC observations"
    )


data["phase5_normal_ihc_state"] = (
    data.apply(
        normal_ihc_state,
        axis=1
    )
)


print("\nNormal-IHC evidence categories:")

print(
    data["phase5_normal_ihc_state"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 10. LOAD HPA TISSUE-LEVEL PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING NORMAL-TISSUE CELL-TYPE SUMMARIES")
print("-" * 65)


tissues = pd.read_csv(
    TISSUE_PATH,
    dtype={"Gene": str},
    low_memory=False
)


required_tissue_columns = {
    "Gene",
    "Tissue",
    "max_level_score",
    "max_reliable_score"
}


if not required_tissue_columns.issubset(
    tissues.columns
):

    raise RuntimeError(
        "Unexpected HPA tissue-level schema."
    )


if tissues[
    ["Gene", "Tissue"]
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene-tissue profiles."
    )


if not set(tissues["Gene"]).issubset(
    set(data["ensembl_stable_id"])
):

    raise RuntimeError(
        "HPA tissue profiles contain "
        "unexpected candidate genes."
    )


for column in [
    "max_level_score",
    "max_reliable_score"
]:

    tissues[column] = pd.to_numeric(
        tissues[column],
        errors="coerce"
    )


print("\nGene-tissue profiles:", len(tissues))

print("Tissues represented:", tissues["Tissue"].nunique())


# ------------------------------------------------------------
# 11. NORMAL COLON, KIDNEY AND SKIN REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL COLON, KIDNEY AND SKIN REVIEW")
print("-" * 65)


FOCUS_TISSUES = [
    "Colon",
    "Kidney",
    "Skin"
]


SCORE_LABELS = {
    0: "Not detected",
    1: "Low",
    2: "Medium",
    3: "High"
}


for tissue_name in FOCUS_TISSUES:

    key = tissue_name.lower()

    subset = tissues[
        tissues["Tissue"]
        .astype(str)
        .str.lower()
        .eq(key)
    ].copy()


    if subset["Gene"].duplicated().any():

        raise RuntimeError(
            f"Duplicate {tissue_name} gene profiles."
        )


    primary_lookup = subset.set_index(
        "Gene"
    )["max_level_score"]


    reliable_lookup = subset.set_index(
        "Gene"
    )["max_reliable_score"]


    score_column = (
        f"phase5_normal_{key}_max_ihc_score"
    )

    reliable_column = (
        f"phase5_normal_{key}_reliable_ihc_score"
    )

    label_column = (
        f"phase5_normal_{key}_ihc_label"
    )


    data[score_column] = (
        data["ensembl_stable_id"]
        .map(primary_lookup)
    )


    data[reliable_column] = (
        data["ensembl_stable_id"]
        .map(reliable_lookup)
    )


    data[label_column] = (
        data[score_column]
        .map(SCORE_LABELS)
        .fillna(
            "No interpretable tissue observation"
        )
    )


    print(
        f"\n{tissue_name}: "
        f"{len(subset)} gene-level profiles"
    )


print("\nImportant:")

print(
    "Tissue-level IHC represents the maximum "
    "reported level among annotated cell types."
)

print(
    "It does not establish clinical toxicity "
    "or accessibility of the protein."
)


# ------------------------------------------------------------
# 12. SEPARATE BIOLOGICAL TARGETING HYPOTHESES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ANNOTATING PUTATIVE TARGETING MECHANISMS")
print("-" * 65)


def targeting_hypothesis(row):

    gene = row["gene_name"]

    category = row[
        "phase5_topology_class"
    ]


    # Explicit biological review hypothesis.
    # This does NOT establish localization
    # to tumor-associated fibroblasts
    # from our bulk datasets.

    if gene == "FAP":

        return (
            "Stromal-associated targeting hypothesis; "
            "tumor cell-type origin requires validation"
        )


    if category in [
        "Membrane with annotated extracellular region",
        "GPI-anchored membrane-associated",
        "Membrane-associated; "
        "extracellular topology unresolved",
        "GPI anchor annotated; "
        "membrane residency requires review"
    ]:

        return (
            "Cell-surface antibody-targeting hypothesis; "
            "cellular origin and access unverified"
        )


    if category == (
        "Transmembrane; "
        "surface localization unresolved"
    ):

        return (
            "Potential membrane target; "
            "plasma-membrane residency unresolved"
        )


    if category == (
        "Secreted/extracellular; "
        "membrane attachment not established"
    ):

        return (
            "Extracellular or microenvironment "
            "targeting hypothesis"
        )


    return (
        "No supported antibody-accessible "
        "mechanism established by current topology"
    )


data["phase5_targeting_hypothesis"] = (
    data.apply(
        targeting_hypothesis,
        axis=1
    )
)


print("\nWorking targeting-hypothesis distribution:")

print(
    data["phase5_targeting_hypothesis"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 13. IDENTIFY EVIDENCE-REVIEW FLAGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING EVIDENCE-REVIEW FLAGS")
print("-" * 65)


def generate_review_flags(row):

    flags = []


    if not row["_uniprot_reviewed"]:

        flags.append(
            "Reviewed UniProt mapping unavailable"
        )


    if not row["_normal_ihc_available"]:

        flags.append(
            "Normal-tissue IHC unavailable"
        )


    if not row["_depmap_available"]:

        flags.append(
            "CRISPR gene-effect measurement unavailable"
        )


    if (
        row["phase5_protein_evidence_state"]
        == "Sparse paired-protein measurements"
    ):

        flags.append(
            "Sparse CPTAC paired-protein coverage"
        )


    if (
        row["phase5_protein_evidence_state"]
        == "No complete MSS paired-protein measurement"
    ):

        flags.append(
            "No complete CPTAC paired-protein observations"
        )


    colon_score = row[
        "phase5_normal_colon_max_ihc_score"
    ]

    kidney_score = row[
        "phase5_normal_kidney_max_ihc_score"
    ]


    if (
        pd.notna(colon_score)
        and colon_score >= 2
    ):

        flags.append(
            "Normal colon medium/high IHC reported"
        )


    if (
        pd.notna(kidney_score)
        and kidney_score >= 2
    ):

        flags.append(
            "Normal kidney medium/high IHC reported"
        )


    if (
        row["gene_name"] == "FAP"
    ):

        flags.append(
            "Stromal-vs-tumor-cell localization review"
        )


    return "; ".join(flags)


data["phase5_evidence_review_flags"] = (
    data.apply(
        generate_review_flags,
        axis=1
    )
)


data["phase5_has_review_flags"] = (
    data["phase5_evidence_review_flags"]
    .ne("")
)


print(
    "\nCandidates with at least one "
    "evidence-review flag:",
    int(
        data["phase5_has_review_flags"].sum()
    )
)


print(
    "Flags indicate questions requiring "
    "review, not candidate disqualification."
)


# ------------------------------------------------------------
# 14. VERIFY SOURCE EVIDENCE WAS PRESERVED
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE EVIDENCE PRESERVATION")
print("-" * 65)


original_columns = source.columns.tolist()


if not source.equals(
    data[original_columns]
):

    raise RuntimeError(
        "Original canonical evidence changed "
        "during classification."
    )


if len(data) != 948:

    raise RuntimeError(
        "Candidate count changed."
    )


if int(
    data["_protein_enriched"].sum()
) != 20:

    raise RuntimeError(
        "Cross-omics support count changed."
    )


if int(
    data["_depmap_available"].sum()
) != 943:

    raise RuntimeError(
        "DepMap coverage changed."
    )


print("\nOriginal evidence columns: UNCHANGED")

print("Candidate count: PASSED")

print("Evidence identity: PASSED")


# ------------------------------------------------------------
# 15. REVIEW 20 CROSS-OMICS-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS-SUPPORTED CANDIDATE REVIEW")
print("-" * 65)


cross_omics = data[
    data["_protein_enriched"]
].copy()


if len(cross_omics) != 20:

    raise RuntimeError(
        "Expected 20 protein-supported genes."
    )


cross_columns = [
    "gene_name",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "hpa_tissues_medium_high",
    "depmap_crc_median_gene_effect"
]


print("\n")

print(
    cross_omics[
        cross_columns
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 16. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING THERAPEUTIC HYPOTHESIS REVIEW")
print("-" * 65)


leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "phase5_topology_class",
    "protein_enriched",
    "mss_complete_pairs",
    "phase5_normal_colon_ihc_label",
    "phase5_normal_kidney_ihc_label",
    "depmap_crc_median_gene_effect"
]


print("\n")

print(
    leading[
        leading_columns
    ].to_string(
        index=False
    )
)


print("\nTargeting hypotheses:")

print(
    leading[
        [
            "gene_name",
            "phase5_targeting_hypothesis"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 17. PREPARE CLASSIFICATION SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CLASSIFICATION SUMMARY")
print("-" * 65)


class_summary = (
    data.groupby(
        "phase5_topology_class",
        dropna=False
    )
    .agg(

        candidates=(
            "gene_id",
            "size"
        ),

        cptac_supported=(
            "_protein_enriched",
            "sum"
        ),

        normal_ihc_available=(
            "_normal_ihc_available",
            "sum"
        ),

        depmap_available=(
            "_depmap_available",
            "sum"
        )

    )
    .reset_index()
)


print("\n")

print(
    class_summary.to_string(
        index=False
    )
)


if int(
    class_summary["candidates"].sum()
) != 948:

    raise RuntimeError(
        "Classification summary lost candidates."
    )


# ------------------------------------------------------------
# 18. PREPARE CLEAN OUTPUT TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CLASSIFICATION OUTPUTS")
print("-" * 65)


# Temporary internal calculation fields
# are not exported.

temporary_columns = [
    column
    for column in data.columns
    if column.startswith("_")
    and column not in original_columns
]


final = data.drop(
    columns=temporary_columns
).copy()


if len(final) != 948:

    raise RuntimeError(
        "Output candidate count changed."
    )


review_flags = final[
    final["phase5_has_review_flags"]
].copy()


focus_tissue_columns = [
    "gene_id",
    "gene_name",
    "phase5_topology_class",
    "phase5_normal_colon_ihc_label",
    "phase5_normal_colon_reliable_ihc_score",
    "phase5_normal_kidney_ihc_label",
    "phase5_normal_kidney_reliable_ihc_score",
    "phase5_normal_skin_ihc_label",
    "phase5_normal_skin_reliable_ihc_score"
]


normal_focus = final[
    focus_tissue_columns
].copy()


surface_groups = final[
    [
        "gene_id",
        "gene_name",
        "phase5_topology_class",
        "phase5_surface_review_group",
        "phase5_targeting_hypothesis",
        "phase5_protein_evidence_state",
        "phase5_normal_ihc_state"
    ]
].copy()


# ------------------------------------------------------------
# 19. SAVE CLASSIFICATION TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING BIOLOGICAL CLASSIFICATION RESULTS")
print("-" * 65)


output_path = (
    RESULTS /
    "surfacetarget_phase5_target_classification_v1.csv"
)


final.to_csv(
    output_path,
    index=False
)


class_summary.to_csv(
    RESULTS /
    "surfacetarget_phase5_topology_class_summary.csv",
    index=False
)


cross_omics.drop(
    columns=temporary_columns,
    errors="ignore"
).to_csv(
    RESULTS /
    "surfacetarget_phase5_crossomics_target_review.csv",
    index=False
)


leading.drop(
    columns=temporary_columns,
    errors="ignore"
).to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_target_review.csv",
    index=False
)


surface_groups.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_review_groups.csv",
    index=False
)


normal_focus.to_csv(
    RESULTS /
    "surfacetarget_phase5_normal_tissue_focus_review.csv",
    index=False
)


review_flags.to_csv(
    RESULTS /
    "surfacetarget_phase5_evidence_review_flags.csv",
    index=False
)


reloaded = pd.read_csv(
    output_path,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved classification table has "
        "an unexpected gene count."
    )


if (
    reloaded["gene_id"].tolist()
    != final["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved candidate identifiers changed."
    )


print("\nAll classification tables saved.")

print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 20. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CLASSIFICATION PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_canonical_table": str(
        CANONICAL_PATH
    ),

    "source_canonical_sha256": actual_hash,

    "hpa_tissue_source": str(
        TISSUE_PATH
    ),

    "hpa_tissue_sha256": sha256_file(
        TISSUE_PATH
    ),

    "candidate_count": int(
        len(final)
    ),

    "protein_supported_candidates": int(
        data["_protein_enriched"].sum()
    ),

    "normal_ihc_annotated_candidates": int(
        data["_normal_ihc_available"].sum()
    ),

    "verified_depmap_mappings": int(
        data["_depmap_available"].sum()
    ),

    "classification_counts": {

        str(row["phase5_topology_class"]):
            int(row["candidates"])

        for _, row in class_summary.iterrows()

    },

    "classification_method": (
        "Rule-based interpretation of existing "
        "UniProt protein-topology categories"
    ),

    "gpi_policy": (
        "GPI anchoring is evaluated separately "
        "from transmembrane segments"
    ),

    "protein_evidence_policy": (
        "Separate qualifying enrichment, "
        "adequate nonqualifying measurements, "
        "sparse coverage and missing observations"
    ),

    "normal_ihc_policy": (
        "Preserve tissue-level staining and "
        "reliability annotations without "
        "assigning clinical safety scores"
    ),

    "targeting_hypothesis_policy": (
        "Working mechanism categories only; "
        "tumor-cell specificity not inferred "
        "from bulk expression"
    ),

    "stromal_hypothesis": (
        "FAP separately flagged for "
        "tumor-stroma localization review"
    ),

    "dependency_policy": (
        "DepMap CRISPR fitness effects retained "
        "as an orthogonal evidence dimension"
    ),

    "original_evidence_preserved": True,

    "candidate_exclusions": 0,

    "therapeutic_target_ranking_performed": False,

    "limitations": [

        "Membrane localization does not "
        "demonstrate antibody accessibility.",

        "Extracellular topology annotations "
        "do not establish binding epitopes.",

        "Bulk tumor enrichment cannot distinguish "
        "malignant from stromal expression.",

        "GPI-anchor annotations do not "
        "prove therapeutic suitability.",

        "Normal-tissue IHC is not "
        "a clinical toxicity measurement.",

        "Maximum tissue staining summarizes "
        "multiple annotated cell types.",

        "Missing protein measurements are "
        "not treated as absent protein.",

        "CRISPR dependency is not required "
        "for antibody-drug conjugate targeting.",

        "The DepMap cohort is not "
        "independently confirmed MSS.",

        "No experimental target validation "
        "has been performed."

    ]
}


with open(
    META /
    "surfacetarget_phase5_classification_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 21. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch52_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.2 SUMMARY")
print("=" * 65)


print("\nCandidate genes:", len(final))

print(
    "Biological topology classes:",
    class_summary.shape[0]
)

print(
    "Cross-omics-supported candidates:",
    len(cross_omics)
)

print(
    "Candidates with normal-IHC evidence:",
    int(
        data["_normal_ihc_available"].sum()
    )
)

print(
    "Verified DepMap mappings:",
    int(
        data["_depmap_available"].sum()
    )
)

print(
    "Candidates with evidence-review flags:",
    len(review_flags)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_target_classification_v1.csv")

print("2. surfacetarget_phase5_topology_class_summary.csv")

print("3. surfacetarget_phase5_crossomics_target_review.csv")

print("4. surfacetarget_phase5_leading_target_review.csv")

print("5. surfacetarget_phase5_surface_review_groups.csv")

print("6. surfacetarget_phase5_normal_tissue_focus_review.csv")

print("7. surfacetarget_phase5_evidence_review_flags.csv")

print("8. surfacetarget_phase5_classification_provenance.json")


print("\nClassification validation: PASSED")

print("Original evidence: PRESERVED")

print("Research checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidate genes excluded.")

print("No clinical safety scores assigned.")

print("No final target rankings performed.")

print("\nPATCH 5.2 COMPLETE")

print("=" * 65)